# Power-law baseline for state-dependent effective liquidity

This notebook is a specification check for the preferred logarithmic
effective-liquidity model. It asks whether the activity-dependent tail term
remains necessary when the central response is a free power law.

The comparison is deliberately matched:

1. pooled logarithmic and pooled power responses (two parameters each);
2. state-dependent logarithmic and state-dependent power responses (four
   parameters each); and
3. an activity-tail extension of each state-dependent response (seven
   parameters each).

All models use the same conditional response cells, weights, activity
variable, full state-specific imbalance support, bootstrap samples and
blocked validation folds. The 4,000-event construction is primary because
its median duration is close to five minutes. This notebook does not replace
the main model unless the power family produces a materially better and more
stable account of the activity-dependent tail.

## Alternative effective-liquidity schedule

Let $u=x/x_{med}$ and $z=\log(\lambda/\lambda_{med})$. The power
benchmark is

$
R_0^P(u,z)=A(z)u^{\delta(z)},
\qquad A(z)=e^{a_0+a_1z},
$

with a bounded exponent

$
\delta(z)=2\frac{e^{d_0+d_1z}}{1+e^{d_0+d_1z}},
\qquad 0<\delta(z)<2.
$

Its marginal response and implied effective liquidity are

$
\frac{\partial R_0^P}{\partial u}
=A(z)\delta(z)u^{\delta(z)-1},
\qquad
\mathcal L_0^P(u,z)
=\frac{u^{1-\delta(z)}}{A(z)\delta(z)}.
$

The same tail adjustment used in the logarithmic model gives

$
\mathcal L^P(u,z)=\mathcal L_0^P(u,z)
\exp\{-\gamma(z-z_*)_+h(u;u_*)\},
\quad
h(u;u_*)=\log\!\left[1+\frac{(u-u_*)_+}{u_*}\right].
$

Above $(u_*), (h(u;u_*)=\log(u/u_*))$, so the power-tail response has an
exact piecewise form. Writing
$\kappa(z)=\gamma(z-z_*)_+$,

$
R^P(u,z)=
\begin{cases}
A(z)u^{\delta(z)}, & u\leq u_*,\\[3pt]
A(z)u_*^{\delta(z)}+
\dfrac{A(z)\delta(z)u_*^{-\kappa(z)}}
      {\delta(z)+\kappa(z)}
\left[u^{\delta(z)+\kappa(z)}-
      u_*^{\delta(z)+\kappa(z)}\right], & u>u_*.
\end{cases}
$

The model reduces exactly to the state-dependent power law when
($\gamma=0$). Unlike the logarithmic benchmark, a concave power law implies
an infinite marginal response as ($u\rightarrow0$); the power family is
therefore primarily a statistical robustness check rather than the preferred
structural interpretation.

In [1]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
from numpy.polynomial.legendre import leggauss
import pandas as pd
from scipy.optimize import least_squares

%matplotlib inline

NOTEBOOK_DIR = Path.cwd()
default_project = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
PROJECT_DIR = Path(os.environ.get('CRYPTO_IMPACT_PROJECT_DIR', default_project)).resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.crypto_impact.impact import add_trailing_impact_normalisation
from src.crypto_impact.monthly_eda import (
    build_fixed_event_bars,
    load_monthly_trades,
    make_time_bars,
)
from src.crypto_impact.regime_analysis import build_analysis_table

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda value: f'{value:,.6f}')

In [2]:
DATA_DIR = PROJECT_DIR / 'data' / 'raw' / 'BTCUSDT_monthly'
FIGURE_DIR = Path(
    os.environ.get('CRYPTO_IMPACT_FIGURE_DIR', PROJECT_DIR / 'reports' / 'figures')
)
TABLE_DIR = Path(
    os.environ.get('CRYPTO_IMPACT_TABLE_DIR', PROJECT_DIR / 'reports' / 'tables')
)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

EVENTS_PER_BAR = int(os.environ.get('CRYPTO_IMPACT_EVENTS_PER_BAR', '4000'))
TIME_WINDOW = '5min'
TRAILING_WINDOW = '1D'
MIN_TRAILING_BARS = 100
N_ACTIVITY_BINS = 5
N_IMBALANCE_BINS = 20
MIN_OBS_PER_CELL = int(os.environ.get('CRYPTO_IMPACT_MIN_OBS_PER_CELL', '50'))
N_QUADRATURE_NODES = 32
N_STARTS = int(os.environ.get('CRYPTO_IMPACT_N_STARTS', '16'))
N_BOOTSTRAP = int(os.environ.get('CRYPTO_IMPACT_N_BOOTSTRAP', '100'))
N_CV_FOLDS = int(os.environ.get('CRYPTO_IMPACT_N_CV_FOLDS', '5'))
MAX_FILES_ENV = int(os.environ.get('CRYPTO_IMPACT_MAX_FILES', '0'))
MAX_FILES = MAX_FILES_ENV or None

OUTPUT_STEM = f'power_baseline_liquidity_{EVENTS_PER_BAR}_event'

files = sorted(DATA_DIR.glob('BTCUSDT-trades-*.parquet'))
if MAX_FILES is not None:
    files = files[:MAX_FILES]
if not files:
    raise FileNotFoundError(f'No monthly parquet files found in {DATA_DIR}')
print(f'Event bars: {EVENTS_PER_BAR:,} events')
print(f'Files selected: {len(files)} ({files[0].name} to {files[-1].name})')

Event bars: 4,000 events
Files selected: 17 (BTCUSDT-trades-2025-01.parquet to BTCUSDT-trades-2026-05.parquet)


## Construct matched response surfaces

The construction and normalisation are identical to the state-dependent
logarithmic notebook. Five activity quintiles are formed, followed by twenty
independent equal-count imbalance bins within each quintile. Every model is
evaluated at the mean imbalance and mean continuous activity observed in the
same 100 cells.

In [6]:
def prepare_normalised_analysis(bars):
    analysis = build_analysis_table(bars, horizons=())
    analysis = add_trailing_impact_normalisation(
        analysis,
        window=TRAILING_WINDOW,
        min_periods=MIN_TRAILING_BARS,
        timestamp_col='end_time',
    ).replace([np.inf, -np.inf], np.nan)
    analysis['block_day'] = pd.to_datetime(
        analysis['end_time'], utc=True
    ).dt.floor('D')
    analysis['event_intensity'] = (
        analysis['trade_count'] / analysis['duration_seconds']
    )
    return analysis


def add_model_scales(data):
    result = data.copy()
    valid = result[[
        'normalised_imbalance', 'normalised_signed_impact', 'event_intensity'
    ]].replace([np.inf, -np.inf], np.nan).dropna()
    valid = valid.loc[
        (valid['normalised_imbalance'] > 0)
        & (valid['event_intensity'] > 0)
    ]
    x_med = float(valid['normalised_imbalance'].median())
    intensity_med = float(valid['event_intensity'].median())
    result['imbalance_u'] = result['normalised_imbalance'] / x_med
    result['activity_z'] = np.log(result['event_intensity'] / intensity_med)
    result = result.replace([np.inf, -np.inf], np.nan).dropna(subset=[
        'imbalance_u', 'normalised_signed_impact', 'activity_z', 'block_day'
    ])
    result = result.loc[result['imbalance_u'] > 0].copy()
    return result, {
        'x_med': x_med,
        'event_intensity_median': intensity_med,
    }


event_bars = build_fixed_event_bars(files, events_per_bar=EVENTS_PER_BAR)
event_analysis, event_scales = add_model_scales(
    prepare_normalised_analysis(event_bars)
)

time_frames = []
for path in files:
    monthly = make_time_bars(load_monthly_trades(path), window=TIME_WINDOW)
    monthly['start_time'] = monthly.index
    monthly['end_time'] = monthly.index + pd.Timedelta(TIME_WINDOW)
    monthly['duration_seconds'] = pd.Timedelta(TIME_WINDOW).total_seconds()
    time_frames.append(monthly)
time_bars = pd.concat(time_frames).sort_index()
if not time_bars.index.is_unique:
    raise ValueError('Five-minute bar start times are not unique.')
time_analysis, time_scales = add_model_scales(
    prepare_normalised_analysis(time_bars)
)

analysis_samples = {
    'event_bars': event_analysis,
    'time_bars': time_analysis,
}
normalisation = pd.DataFrame({
    'event_bars': event_scales,
    'time_bars': time_scales,
}).T
normalisation.index.name = 'construction'

normalisation

,x_med,event_intensity_median
construction,,
event_bars,0.000540,13.085601
time_bars,0.000358,7.186667


In [7]:
def build_response_surface(data):
    columns = [
        'imbalance_u', 'activity_z',
        'normalised_signed_impact', 'block_day',
    ]
    sample = data.loc[:, columns].replace([np.inf, -np.inf], np.nan).dropna()
    sample = sample.loc[sample['imbalance_u'] > 0].copy()
    sample['activity_bin'] = pd.qcut(
        sample['activity_z'], q=N_ACTIVITY_BINS,
        labels=False, duplicates='drop',
    )
    if sample['activity_bin'].nunique() != N_ACTIVITY_BINS:
        raise ValueError('Insufficient distinct activity values for five quintiles.')

    frames = []
    for activity_bin, group in sample.groupby('activity_bin', observed=True):
        group = group.copy()
        group['imbalance_bin'] = pd.qcut(
            group['imbalance_u'], q=N_IMBALANCE_BINS,
            labels=False, duplicates='drop',
        )
        if group['imbalance_bin'].nunique() != N_IMBALANCE_BINS:
            raise ValueError(
                f'Insufficient distinct imbalance values in activity bin {activity_bin}.'
            )
        cell = (
            group.groupby('imbalance_bin', observed=True)
            .agg(
                mean_u=('imbalance_u', 'mean'),
                median_u=('imbalance_u', 'median'),
                mean_z=('activity_z', 'mean'),
                median_z=('activity_z', 'median'),
                mean_impact=('normalised_signed_impact', 'mean'),
                median_impact=('normalised_signed_impact', 'median'),
                impact_std=('normalised_signed_impact', 'std'),
                n_obs=('normalised_signed_impact', 'count'),
                n_days=('block_day', 'nunique'),
            )
            .reset_index()
        )
        cell.insert(0, 'activity_bin', int(activity_bin))
        frames.append(cell)

    surface = pd.concat(frames, ignore_index=True)
    surface['standard_error'] = surface['impact_std'] / np.sqrt(surface['n_obs'])
    surface = surface.loc[
        (surface['n_obs'] >= MIN_OBS_PER_CELL)
        & np.isfinite(surface['standard_error'])
        & (surface['standard_error'] > 0)
    ].copy()
    surface['activity_bin'] = surface['activity_bin'].astype(int)
    surface['imbalance_bin'] = surface['imbalance_bin'].astype(int)
    return surface.sort_values(['activity_bin', 'imbalance_bin'])


surfaces = {
    construction: build_response_surface(sample)
    for construction, sample in analysis_samples.items()
}

pd.DataFrame({
    construction: {
        'observations': len(analysis_samples[construction]),
        'UTC days': analysis_samples[construction]['block_day'].nunique(),
        'surface cells': len(surface),
        'minimum observations per retained cell': surface['n_obs'].min(),
    }
    for construction, surface in surfaces.items()
}).T

,observations,UTC days,surface cells,minimum observations per retained cell
event_bars,111576,506,100,1115
time_bars,148505,517,100,1484


## Matched logarithmic and power models

The exponent transform permits both concave and convex power responses while
retaining the earlier (0<\delta<2) bounds. The logarithmic tail is evaluated
by deterministic Gauss--Legendre quadrature. The power-tail expression is
evaluated analytically. Multiple deterministic starts are used for every fit.

In [8]:
QUAD_NODES, QUAD_WEIGHTS = leggauss(N_QUADRATURE_NODES)
MODEL_ORDER = [
    'pooled_log', 'pooled_power',
    'state_log', 'state_power',
    'log_activity_tail', 'power_activity_tail',
]
PARAMETER_NAMES = {
    'pooled_log': ['log_A', 'log_u0'],
    'pooled_power': ['log_A', 'delta_logit'],
    'state_log': ['log_A', 'A_activity_slope', 'log_u0', 'u0_activity_slope'],
    'state_power': [
        'log_A', 'A_activity_slope', 'delta_logit', 'delta_activity_slope'
    ],
    'log_activity_tail': [
        'log_A', 'A_activity_slope', 'log_u0', 'u0_activity_slope',
        'gamma_activity', 'activity_threshold_z', 'log_imbalance_threshold_u',
    ],
    'power_activity_tail': [
        'log_A', 'A_activity_slope', 'delta_logit', 'delta_activity_slope',
        'gamma_activity', 'activity_threshold_z', 'log_imbalance_threshold_u',
    ],
}


def logistic(value):
    value = np.asarray(value, dtype=float)
    return 1.0 / (1.0 + np.exp(-np.clip(value, -40.0, 40.0)))


def inverse_delta_transform(delta):
    probability = np.clip(delta / 2.0, 1e-6, 1.0 - 1e-6)
    return np.log(probability / (1.0 - probability))


def log_central_parameters(params, z):
    return (
        np.exp(params[0] + params[1] * z),
        np.exp(params[2] + params[3] * z),
    )


def power_central_parameters(params, z):
    return (
        np.exp(params[0] + params[1] * z),
        2.0 * logistic(params[2] + params[3] * z),
    )


def logarithmic_tail_response(u, z, params):
    u = np.asarray(u, dtype=float)
    z = np.asarray(z, dtype=float)
    A, u0 = log_central_parameters(params, z)
    gamma, z_star, log_u_star = params[4:7]
    u_star = np.exp(log_u_star)
    kappa = gamma * np.maximum(z - z_star, 0.0)
    central_end = np.minimum(u, u_star)
    central = A * np.log1p(central_end / u0)
    width = np.maximum(u - u_star, 0.0)
    nodes = u_star + 0.5 * width[:, None] * (QUAD_NODES[None, :] + 1.0)
    h = np.log(nodes / u_star)
    integrand = (
        A[:, None] * np.exp(np.clip(kappa[:, None] * h, 0.0, 30.0))
        / (u0[:, None] + nodes)
    )
    tail = 0.5 * width * np.sum(QUAD_WEIGHTS[None, :] * integrand, axis=1)
    exact_tail = A * (np.log1p(u / u0) - np.log1p(central_end / u0))
    tail = np.where(kappa <= 1e-12, exact_tail, tail)
    return central + tail


def power_tail_response(u, z, params):
    u = np.asarray(u, dtype=float)
    z = np.asarray(z, dtype=float)
    A, delta = power_central_parameters(params, z)
    gamma, z_star, log_u_star = params[4:7]
    u_star = np.exp(log_u_star)
    kappa = gamma * np.maximum(z - z_star, 0.0)
    below = A * u**delta
    exponent = delta + kappa
    above = (
        A * u_star**delta
        + A * delta * u_star**(-kappa)
        * (u**exponent - u_star**exponent) / exponent
    )
    return np.where(u <= u_star, below, above)


def predict_model(model, params, data):
    u = data['mean_u'].to_numpy(dtype=float)
    z = data['mean_z'].to_numpy(dtype=float)
    if model == 'pooled_log':
        A, u0 = np.exp(params)
        return A * np.log1p(u / u0)
    if model == 'pooled_power':
        A = np.exp(params[0])
        delta = 2.0 * logistic(params[1])
        return A * u**delta
    if model == 'state_log':
        A, u0 = log_central_parameters(params, z)
        return A * np.log1p(u / u0)
    if model == 'state_power':
        A, delta = power_central_parameters(params, z)
        return A * u**delta
    if model == 'log_activity_tail':
        return logarithmic_tail_response(u, z, params)
    if model == 'power_activity_tail':
        return power_tail_response(u, z, params)
    raise ValueError(f'Unknown model: {model}')


def parameter_bounds(model, surface):
    z = surface['mean_z'].to_numpy()
    u = surface['mean_u'].to_numpy()
    if model == 'pooled_log':
        return np.array([-12.0, -7.0]), np.array([2.0, 7.0])
    if model == 'pooled_power':
        return np.array([-12.0, -10.0]), np.array([2.0, 10.0])
    if model == 'state_log':
        return np.array([-12.0, -3.0, -7.0, -3.0]), np.array([2.0, 3.0, 7.0, 3.0])
    if model == 'state_power':
        return np.array([-12.0, -3.0, -10.0, -5.0]), np.array([2.0, 3.0, 10.0, 5.0])
    if model == 'log_activity_tail':
        central_lower = [-12.0, -3.0, -7.0, -3.0]
        central_upper = [2.0, 3.0, 7.0, 3.0]
    elif model == 'power_activity_tail':
        central_lower = [-12.0, -3.0, -10.0, -5.0]
        central_upper = [2.0, 3.0, 10.0, 5.0]
    else:
        raise ValueError(f'Unknown model: {model}')
    return (
        np.array([
            *central_lower, 0.0, np.quantile(z, 0.10),
            np.log(np.quantile(u, 0.40)),
        ]),
        np.array([
            *central_upper, 5.0, np.quantile(z, 0.95),
            np.log(np.quantile(u, 0.995)),
        ]),
    )


def initial_parameters(model, surface, fits):
    positive = surface.loc[surface['mean_impact'] > 0]
    if positive.empty:
        A0, delta0 = 0.03, 0.6
        impact_median = A0
    else:
        log_u = np.log(positive['mean_u'].to_numpy())
        log_y = np.log(positive['mean_impact'].to_numpy())
        delta0, log_A0 = np.polyfit(log_u, log_y, 1)
        delta0 = float(np.clip(delta0, 0.05, 1.95))
        A0 = float(np.exp(log_A0))
        impact_median = float(positive['mean_impact'].median())
    if model == 'pooled_log':
        return np.log([max(impact_median, 1e-4), 1.0])
    if model == 'pooled_power':
        return np.array([np.log(max(A0, 1e-4)), inverse_delta_transform(delta0)])
    if model == 'state_log':
        pooled = fits['pooled_log']['params']
        return np.array([pooled[0], 0.0, pooled[1], 0.0])
    if model == 'state_power':
        pooled = fits['pooled_power']['params']
        return np.array([pooled[0], 0.0, pooled[1], 0.0])
    if model == 'log_activity_tail':
        central = fits['state_log']['params']
    elif model == 'power_activity_tail':
        central = fits['state_power']['params']
    else:
        raise ValueError(f'Unknown model: {model}')
    return np.array([
        *central, 0.15,
        surface['mean_z'].quantile(0.80),
        np.log(surface['mean_u'].quantile(0.90)),
    ])


def fit_one_model(model, surface, fits, n_starts=N_STARTS, random_state=0):
    data = surface.replace([np.inf, -np.inf], np.nan).dropna(subset=[
        'mean_u', 'mean_z', 'mean_impact', 'standard_error'
    ]).copy()
    lower, upper = parameter_bounds(model, data)
    initial = np.clip(
        initial_parameters(model, data, fits), lower + 1e-8, upper - 1e-8
    )
    sigma = data['standard_error'].to_numpy(dtype=float)
    fallback = np.median(sigma[np.isfinite(sigma) & (sigma > 0)])
    sigma = np.where(np.isfinite(sigma) & (sigma > 0), sigma, fallback)
    y = data['mean_impact'].to_numpy(dtype=float)

    def residuals(params):
        return (y - predict_model(model, params, data)) / sigma

    rng = np.random.default_rng(random_state)
    starts = [initial]
    for _ in range(max(1, n_starts) - 1):
        draw = initial + rng.normal(0.0, 0.15, len(initial)) * (upper - lower)
        starts.append(np.clip(draw, lower + 1e-7, upper - 1e-7))

    best = None
    for start in starts:
        result = least_squares(
            residuals, start, bounds=(lower, upper), max_nfev=25_000,
            xtol=1e-10, ftol=1e-10, gtol=1e-10,
        )
        if best is None or np.sum(result.fun**2) < np.sum(best.fun**2):
            best = result
    prediction = predict_model(model, best.x, data)
    return {
        'model': model,
        'params': best.x,
        'lower_bounds': lower,
        'upper_bounds': upper,
        'success': bool(best.success),
        'cost': float(np.sum(best.fun**2)),
        'data': data,
        'prediction': prediction,
        'residual': y - prediction,
    }


def fit_model_sequence(surface, models=MODEL_ORDER, n_starts=N_STARTS, random_state=0):
    fits = {}
    for model_number, model in enumerate(models):
        fits[model] = fit_one_model(
            model, surface, fits, n_starts=n_starts,
            random_state=random_state + model_number,
        )
    return fits

In [9]:
model_fits = {
    construction: fit_model_sequence(
        surface, random_state=10_000 + construction_number * 100
    )
    for construction_number, (construction, surface) in enumerate(surfaces.items())
}

## In-sample model comparison

The direct comparison of interest is between the seven-parameter logarithmic
and power activity-tail models. Because the parameter counts are identical,
their weighted chi-square and AIC differences have the same ordering. The
nested comparisons within each family show whether the tail term is still
required after activity is allowed to change the central curve.

In [10]:
def weighted_rmse(residual, standard_error):
    weights = 1.0 / np.asarray(standard_error, dtype=float)**2
    residual = np.asarray(residual, dtype=float)
    return float(np.sqrt(np.sum(weights * residual**2) / np.sum(weights)))


def fit_summary_row(fit):
    data = fit['data']
    residual = fit['residual']
    tail = data['imbalance_bin'] == data['imbalance_bin'].max()
    high_tail = tail & (data['activity_bin'] == data['activity_bin'].max())
    k = len(fit['params'])
    tolerance = 1e-4 * (fit['upper_bounds'] - fit['lower_bounds'])
    at_bound = (
        (fit['params'] - fit['lower_bounds'] <= tolerance)
        | (fit['upper_bounds'] - fit['params'] <= tolerance)
    )
    names = np.asarray(PARAMETER_NAMES[fit['model']])[at_bound]
    return {
        'model': fit['model'],
        'parameters': k,
        'cells': len(data),
        'weighted_rmse': weighted_rmse(residual, data['standard_error']),
        'tail_wrmse': weighted_rmse(
            residual[tail], data.loc[tail, 'standard_error']
        ),
        'high_activity_tail_wrmse': weighted_rmse(
            residual[high_tail], data.loc[high_tail, 'standard_error']
        ),
        'unweighted_rmse': float(np.sqrt(np.mean(residual**2))),
        'weighted_chi_square': fit['cost'],
        'weighted_aic_up_to_constant': fit['cost'] + 2 * k,
        'parameters_at_bound': int(at_bound.sum()),
        'bound_parameter_names': ';'.join(names),
        'converged': fit['success'],
    }


def transformed_parameters(model, params, surface):
    output = {'model': model}
    if 'power' in model:
        output.update({
            'A_at_median_activity': np.exp(params[0]),
            'A_activity_slope': params[1] if model != 'pooled_power' else np.nan,
            'delta_at_median_activity': 2.0 * logistic(
                params[2] if model != 'pooled_power' else params[1]
            ),
            'delta_activity_slope_logit': (
                params[3] if model not in {'pooled_power'} else np.nan
            ),
        })
        if model != 'pooled_power':
            for label, z_value in [
                ('p10', surface['mean_z'].quantile(0.10)),
                ('p90', surface['mean_z'].quantile(0.90)),
            ]:
                output[f'delta_at_activity_{label}'] = (
                    2.0 * logistic(params[2] + params[3] * z_value)
                )
    else:
        if model == 'pooled_log':
            output.update({
                'A_at_median_activity': np.exp(params[0]),
                'u0_at_median_activity': np.exp(params[1]),
            })
        else:
            output.update({
                'A_at_median_activity': np.exp(params[0]),
                'A_activity_slope': params[1],
                'u0_at_median_activity': np.exp(params[2]),
                'u0_activity_slope': params[3],
            })
    if model in {'log_activity_tail', 'power_activity_tail'}:
        output.update({
            'gamma_activity': params[4],
            'activity_threshold_z': params[5],
            'activity_threshold_multiple': np.exp(params[5]),
            'imbalance_threshold_u': np.exp(params[6]),
        })
    return output


fit_rows = []
parameter_rows = []
prediction_frames = []
for construction, fits in model_fits.items():
    for model, fit in fits.items():
        fit_rows.append({
            'events_per_event_bar': EVENTS_PER_BAR,
            'construction': construction,
            **fit_summary_row(fit),
        })
        parameter_rows.append({
            'events_per_event_bar': EVENTS_PER_BAR,
            'construction': construction,
            **transformed_parameters(model, fit['params'], surfaces[construction]),
        })
        prediction = fit['data'].copy()
        prediction.insert(0, 'construction', construction)
        prediction['model'] = model
        prediction['predicted_impact'] = fit['prediction']
        prediction['residual'] = fit['residual']
        prediction_frames.append(prediction)

fit_statistics = pd.DataFrame(fit_rows)
parameter_estimates = pd.DataFrame(parameter_rows)
surface_predictions = pd.concat(prediction_frames, ignore_index=True)
fit_statistics

,events_per_event_bar,construction,model,parameters,cells,weighted_rmse,tail_wrmse,high_activity_tail_wrmse,unweighted_rmse,weighted_chi_square,weighted_aic_up_to_constant,parameters_at_bound,bound_parameter_names,converged
0,4000,event_bars,pooled_log,2,100,0.002635,0.005055,0.011425,0.002869,359.182914,363.182914,0,,True
1,4000,event_bars,pooled_power,2,100,0.003492,0.009585,0.010441,0.003884,630.482151,634.482151,0,,True
2,4000,event_bars,state_log,4,100,0.001733,0.004893,0.012622,0.002141,155.298251,163.298251,0,,True
3,4000,event_bars,state_power,4,100,0.002901,0.009216,0.006460,0.003280,435.197907,443.197907,0,,True
4,4000,event_bars,log_activity_tail,7,100,0.001395,0.002465,0.004026,0.001489,100.621362,114.621362,1,log_imbalance_threshold_u,True
5,4000,event_bars,power_activity_tail,7,100,0.002901,0.009216,0.006460,0.003280,435.197907,449.197907,1,gamma_activity,True
6,4000,time_bars,pooled_log,2,100,0.003560,0.017967,0.073903,0.009326,"1,742.781360","1,746.781360",0,,True
7,4000,time_bars,pooled_power,2,100,0.003369,0.015829,0.011513,0.005646,"1,560.261648","1,564.261648",0,,True
8,4000,time_bars,state_log,4,100,0.000953,0.002913,0.017521,0.002060,124.820393,132.820393,0,,True
9,4000,time_bars,state_power,4,100,0.002020,0.007618,0.007336,0.002775,561.227775,569.227775,0,,True


In [11]:
parameter_estimates

,events_per_event_bar,construction,model,A_at_median_activity,u0_at_median_activity,A_activity_slope,delta_at_median_activity,delta_activity_slope_logit,u0_activity_slope,delta_at_activity_p10,delta_at_activity_p90,gamma_activity,activity_threshold_z,activity_threshold_multiple,imbalance_threshold_u
0,4000,event_bars,pooled_log,0.054417,1.214985,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4000,event_bars,pooled_power,0.032252,NaN,NaN,0.633893,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4000,event_bars,state_log,0.054155,1.206041,-0.098270,NaN,NaN,-0.256275,NaN,NaN,NaN,NaN,NaN,NaN
3,4000,event_bars,state_power,0.032086,NaN,0.084042,0.637392,-0.101281,NaN,0.680414,0.576936,NaN,NaN,NaN,NaN
4,4000,event_bars,log_activity_tail,0.043026,0.866649,-0.309118,NaN,NaN,-0.585347,NaN,NaN,0.133978,-0.950877,0.386402,0.756493
5,4000,event_bars,power_activity_tail,0.032086,NaN,0.084042,0.637392,-0.101281,NaN,0.680414,0.576936,0.000000,0.906980,2.476832,5.207456
6,4000,time_bars,pooled_log,0.056051,2.112849,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,4000,time_bars,pooled_power,0.021757,NaN,NaN,0.690970,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,4000,time_bars,state_log,0.037567,1.191099,0.554173,NaN,NaN,0.608014,NaN,NaN,NaN,NaN,NaN,NaN
9,4000,time_bars,state_power,0.022577,NaN,0.185174,0.633961,-0.025087,NaN,0.642943,0.622333,NaN,NaN,NaN,NaN


## Day-block bootstrap

Complete UTC days are resampled and the four state-dependent models are
refitted. The bootstrap records the tail coefficient under each central
response family and the direct difference in fit between the two matched
seven-parameter tail models.

In [ ]:
BOOTSTRAP_MODELS = [
    'state_log', 'state_power',
    'log_activity_tail', 'power_activity_tail',
]


def bootstrap_comparison(data, n_bootstrap=N_BOOTSTRAP, random_state=0):
    day_groups = {
        day: group for day, group in data.groupby('block_day', observed=True)
    }
    days = np.asarray(list(day_groups), dtype=object)
    if len(days) < 2:
        raise ValueError('At least two UTC days are required for the bootstrap.')
    rng = np.random.default_rng(random_state)
    rows = []
    for bootstrap_sample in range(n_bootstrap):
        sampled_days = rng.choice(days, size=len(days), replace=True)
        draw = pd.concat(
            [day_groups[day].assign(_bootstrap_day=draw_number)
             for draw_number, day in enumerate(sampled_days)],
            ignore_index=True,
        )
        try:
            surface = build_response_surface(draw)
            # Pooled models are required only to initialise the state models.
            fits = fit_model_sequence(
                surface,
                models=MODEL_ORDER,
                n_starts=max(3, N_STARTS // 3),
                random_state=random_state + bootstrap_sample * 10,
            )
        except (RuntimeError, ValueError):
            continue
        summaries = {model: fit_summary_row(fits[model]) for model in BOOTSTRAP_MODELS}
        log_tail = transformed_parameters(
            'log_activity_tail', fits['log_activity_tail']['params'], surface
        )
        power_tail = transformed_parameters(
            'power_activity_tail', fits['power_activity_tail']['params'], surface
        )
        rows.append({
            'bootstrap_sample': bootstrap_sample,
            'log_gamma_activity': log_tail['gamma_activity'],
            'power_gamma_activity': power_tail['gamma_activity'],
            'log_tail_wrmse_improvement_pct': 100 * (
                summaries['state_log']['tail_wrmse']
                - summaries['log_activity_tail']['tail_wrmse']
            ) / summaries['state_log']['tail_wrmse'],
            'power_tail_wrmse_improvement_pct': 100 * (
                summaries['state_power']['tail_wrmse']
                - summaries['power_activity_tail']['tail_wrmse']
            ) / summaries['state_power']['tail_wrmse'],
            'power_minus_log_tail_wrmse': (
                summaries['power_activity_tail']['tail_wrmse']
                - summaries['log_activity_tail']['tail_wrmse']
            ),
            'power_minus_log_overall_wrmse': (
                summaries['power_activity_tail']['weighted_rmse']
                - summaries['log_activity_tail']['weighted_rmse']
            ),
        })
    return pd.DataFrame(rows)


bootstrap_frames = []
for construction_number, (construction, sample) in enumerate(analysis_samples.items()):
    draws = bootstrap_comparison(
        sample, n_bootstrap=N_BOOTSTRAP,
        random_state=20_000 + construction_number * 1_000,
    )
    draws.insert(0, 'construction', construction)
    bootstrap_frames.append(draws)
bootstrap_draws = pd.concat(bootstrap_frames, ignore_index=True)

bootstrap_summary = (
    bootstrap_draws.groupby('construction', observed=True)
    .agg(
        bootstrap_samples=('bootstrap_sample', 'count'),
        log_gamma_median=('log_gamma_activity', 'median'),
        log_gamma_p025=('log_gamma_activity', lambda values: values.quantile(0.025)),
        log_gamma_p975=('log_gamma_activity', lambda values: values.quantile(0.975)),
        power_gamma_median=('power_gamma_activity', 'median'),
        power_gamma_p025=('power_gamma_activity', lambda values: values.quantile(0.025)),
        power_gamma_p975=('power_gamma_activity', lambda values: values.quantile(0.975)),
        probability_positive_power_tail_improvement=(
            'power_tail_wrmse_improvement_pct', lambda values: (values > 0).mean()
        ),
        probability_power_tail_beats_log_tail=(
            'power_minus_log_tail_wrmse', lambda values: (values < 0).mean()
        ),
        median_power_minus_log_tail_wrmse=('power_minus_log_tail_wrmse', 'median'),
    )
)
bootstrap_summary

## Blocked validation

Contiguous UTC-day blocks are held out in turn. This is blocked validation,
not a rolling forecasting exercise: the training data may lie on both sides
of the held-out block. Surfaces are rebuilt separately in the training and
test samples, and the test cells are scored without refitting.

In [ ]:
def blocked_cross_validation(data, n_folds=N_CV_FOLDS, random_state=0):
    days = np.asarray(sorted(data['block_day'].dropna().unique()))
    if len(days) < n_folds:
        raise ValueError('Fewer UTC days than requested validation folds.')
    rows = []
    for fold_number, test_days in enumerate(np.array_split(days, n_folds)):
        train = data.loc[~data['block_day'].isin(test_days)]
        test = data.loc[data['block_day'].isin(test_days)]
        train_surface = build_response_surface(train)
        test_surface = build_response_surface(test)
        fits = fit_model_sequence(
            train_surface,
            n_starts=max(4, N_STARTS // 2),
            random_state=random_state + fold_number * 100,
        )
        for model, fit in fits.items():
            prediction = predict_model(model, fit['params'], test_surface)
            residual = test_surface['mean_impact'].to_numpy() - prediction
            tail = test_surface['imbalance_bin'] == test_surface['imbalance_bin'].max()
            rows.append({
                'fold': fold_number,
                'test_start': pd.Timestamp(test_days[0]),
                'test_end': pd.Timestamp(test_days[-1]),
                'model': model,
                'test_cells': len(test_surface),
                'test_weighted_rmse': weighted_rmse(
                    residual, test_surface['standard_error']
                ),
                'test_tail_weighted_rmse': weighted_rmse(
                    residual[tail], test_surface.loc[tail, 'standard_error']
                ),
            })
    return pd.DataFrame(rows)


validation_frames = []
for construction_number, (construction, sample) in enumerate(analysis_samples.items()):
    validation = blocked_cross_validation(
        sample, n_folds=N_CV_FOLDS,
        random_state=30_000 + construction_number * 1_000,
    )
    validation.insert(0, 'construction', construction)
    validation.insert(0, 'events_per_event_bar', EVENTS_PER_BAR)
    validation_frames.append(validation)
cross_validation = pd.concat(validation_frames, ignore_index=True)

cross_validation_summary = (
    cross_validation.groupby(['construction', 'model'], observed=True)
    .agg(
        folds=('fold', 'count'),
        mean_test_wrmse=('test_weighted_rmse', 'mean'),
        mean_test_tail_wrmse=('test_tail_weighted_rmse', 'mean'),
    )
    .reset_index()
)
cross_validation_summary

## Direct graphical comparison

The upper panels compare the two seven-parameter models in the fastest
activity quintile. Lines are predicted cell paths, evaluated at every cell's
own mean activity. The lower panels plot observed minus fitted values. The
figure is diagnostic: model selection should be based on the complete cells,
bootstrap and blocked validation rather than only this quintile.

In [ ]:
fig, axes = plt.subplots(
    2, 2, figsize=(13.0, 8.0), sharex='col', constrained_layout=True,
    gridspec_kw={'height_ratios': [1.7, 1.0]},
)
model_styles = {
    'log_activity_tail': ('#0072B2', '-', 'Log activity-tail'),
    'power_activity_tail': ('#D55E00', '--', 'Power activity-tail'),
}

for column, construction in enumerate(['event_bars', 'time_bars']):
    surface = surfaces[construction]
    state = surface.loc[
        surface['activity_bin'] == surface['activity_bin'].max()
    ].sort_values('mean_u')
    top, bottom = axes[0, column], axes[1, column]
    top.errorbar(
        state['mean_u'], state['mean_impact'], yerr=state['standard_error'],
        fmt='o', ms=4, capsize=1.5, color='black', alpha=0.60,
        label='Observed cell mean',
    )
    for model, (colour, linestyle, label) in model_styles.items():
        prediction = predict_model(model, model_fits[construction][model]['params'], state)
        residual = state['mean_impact'].to_numpy() - prediction
        top.plot(
            state['mean_u'], prediction, color=colour,
            ls=linestyle, lw=1.8, label=label,
        )
        bottom.plot(
            state['mean_u'], residual, marker='o', ms=3.2,
            color=colour, ls=linestyle, lw=1.4, label=label,
        )
    title = (
        f'{EVENTS_PER_BAR:,}-event bars'
        if construction == 'event_bars' else 'Five-minute bars'
    )
    top.set_title(f'{title}: fastest activity quintile')
    top.set_xscale('log')
    top.set_ylabel('Normalized signed impact')
    top.grid(alpha=0.15, linewidth=0.5)
    bottom.axhline(0, color='black', lw=0.7, alpha=0.6)
    bottom.set_xscale('log')
    bottom.set_xlabel(r'Normalized imbalance $u=x/x_{med}$')
    bottom.set_ylabel('Observed minus fitted')
    bottom.grid(alpha=0.15, linewidth=0.5)

axes[0, 0].legend(frameon=False, fontsize=8)
figure_path = FIGURE_DIR / f'20_{OUTPUT_STEM}_comparison.pdf'
fig.savefig(figure_path, bbox_inches='tight')
plt.show()

## Save outputs and decision table

In [ ]:
decision_rows = []
for construction in ['event_bars', 'time_bars']:
    fits = fit_statistics.loc[
        fit_statistics['construction'] == construction
    ].set_index('model')
    validation = cross_validation_summary.loc[
        cross_validation_summary['construction'] == construction
    ].set_index('model')
    for family, state_model, tail_model in [
        ('logarithmic', 'state_log', 'log_activity_tail'),
        ('power', 'state_power', 'power_activity_tail'),
    ]:
        decision_rows.append({
            'construction': construction,
            'family': family,
            'tail_gamma': parameter_estimates.loc[
                (parameter_estimates['construction'] == construction)
                & (parameter_estimates['model'] == tail_model),
                'gamma_activity',
            ].iloc[0],
            'in_sample_wrmse_improvement_pct': 100 * (
                fits.loc[state_model, 'weighted_rmse']
                - fits.loc[tail_model, 'weighted_rmse']
            ) / fits.loc[state_model, 'weighted_rmse'],
            'in_sample_tail_wrmse_improvement_pct': 100 * (
                fits.loc[state_model, 'tail_wrmse']
                - fits.loc[tail_model, 'tail_wrmse']
            ) / fits.loc[state_model, 'tail_wrmse'],
            'blocked_test_wrmse': validation.loc[tail_model, 'mean_test_wrmse'],
            'blocked_test_tail_wrmse': validation.loc[
                tail_model, 'mean_test_tail_wrmse'
            ],
        })
decision_table = pd.DataFrame(decision_rows)
decision_table

In [ ]:
normalisation.to_csv(TABLE_DIR / f'{OUTPUT_STEM}_normalisation.csv')
fit_statistics.to_csv(TABLE_DIR / f'{OUTPUT_STEM}_fit_statistics.csv', index=False)
parameter_estimates.to_csv(TABLE_DIR / f'{OUTPUT_STEM}_parameters.csv', index=False)
surface_predictions.to_csv(
    TABLE_DIR / f'{OUTPUT_STEM}_surface_predictions.csv', index=False
)
bootstrap_draws.to_csv(TABLE_DIR / f'{OUTPUT_STEM}_bootstrap.csv', index=False)
bootstrap_summary.to_csv(TABLE_DIR / f'{OUTPUT_STEM}_bootstrap_summary.csv')
cross_validation.to_csv(
    TABLE_DIR / f'{OUTPUT_STEM}_cross_validation.csv', index=False
)
cross_validation_summary.to_csv(
    TABLE_DIR / f'{OUTPUT_STEM}_cross_validation_summary.csv', index=False
)
decision_table.to_csv(TABLE_DIR / f'{OUTPUT_STEM}_decision_table.csv', index=False)

print('Figure:', figure_path)
print('Tables:', TABLE_DIR / f'{OUTPUT_STEM}_*.csv')

## Interpretation checklist

The activity-dependent liquidity mechanism is robust to the central response
only if the power construction satisfies all of the following:

1. the power-tail estimate of (\gamma) is economically non-negligible and
   stable under UTC-day resampling;
2. the power tail materially improves on the state-dependent power response,
   particularly in held-out tail cells;
3. threshold and exponent parameters are not repeatedly pinned at their
   bounds;
4. the result is present under both event and time bars; and
5. any advantage over the matched logarithmic tail persists in blocked
   validation rather than only in sample.

If the power model removes the need for (\gamma), the logarithmic tail term
should be interpreted primarily as a functional-form correction. If both
central families require a positive tail term, the evidence for a separate
activity-dependent liquidity shortfall is substantially stronger. Even then,
neither model identifies separate order-arrival and cancellation rates.